# Tahap 3: Model Training (Baselines & Machine Learning)
### Pelatihan Bertahap Model Prediksi Karhutla Tanpa Risiko Kehabisan Memori (Out-of-Core / Stratified Streaming)

---

## 📌 Alur Pemodelan Berdasarkan `Bluprin.md`

Sesuai Section 4 Blueprint, pemodelan dibangun secara bertahap dari model paling sederhana hingga machine learning:

```
Baseline 0: Persistence Baseline (Kondisi besok diasumsikan sama dengan hari ini)
      ↓
Baseline 1: Markov Chain (First-Order Discrete Transition Matrix)
      ↓
Model 2A: Supervised Linear (Streaming Logistic Regression via Online SGD)
      ↓
Model 2B: Supervised Non-Linear (Memory-Efficient LightGBM with Stratified Subsampling)
```

### 🔒 Aturan Validasi Temporal (*Strict Out-of-Time Split*):
- **Training Set**: 22 September 2016 – 31 Desember 2022 (~6,25 tahun, ~12,3 juta observasi)
- **Validation Set**: 1 Januari 2023 – 31 Desember 2024 (2 tahun, ~3,95 juta observasi)
- **Test Set**: 1 Januari 2025 – 31 Agustus 2026 (~1,67 tahun, ~3,3 juta observasi)
- **Batasan Mutlak**: $t_{train} < t_{val} < t_{test}$, tidak ada informasi masa depan yang bocor ke pelatihan (*zero data leakage*).


In [ ]:
import sys
import os
import glob
from pathlib import Path
import psutil
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Hubungkan ke modul lokal project
PROJECT_ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    FEATURE_COLS, TARGET_COL, MODELS_DIR, TRAIN_END, VAL_END, TEST_END
)
from src.preprocessing.batch_builder import iter_panel_batches
from src.models.baselines import PersistenceBaseline, MarkovChainBaseline
from src.models.train_ml import StreamingLogisticRegression, train_lightgbm_memory_efficient

MODELS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Direktori Simpan Model: {MODELS_DIR}")


---
## 1. Model 0 — Persistence Baseline

$$\hat{S}_{g, t+1} = S_{g, t}$$

Kondisi besok diasumsikan sama persis dengan kondisi hari ini:
- Jika hari ini terbakar ($S_t = 1$), prediksi besok adalah 1.
- Jika hari ini padam ($S_t = 0$), prediksi besok adalah 0.

Persistence tidak memerlukan training, namun menjadi patokan (*benchmark*) absolut: **Apakah model statistik atau ML yang lebih rumit benar-benar memberikan peningkatan performa dibanding asumsi naif ini?**


In [ ]:
# Inisialisasi Persistence Baseline
model_persistence = PersistenceBaseline()
print(f"Model 0 siap: {model_persistence.name}")


---
## 2. Model 1 — First-Order Discrete Markov Chain

Model Rantai Markov menghitung probabilitas transisi empiris:

$$P(S_{t+1} \mid S_t)$$

Model ini dilatih secara **streaming akumulatif** melintasi batch-batch data training (2016 s.d. 2022) dengan memperbarui matriks frekuensi transisi:
$$N_{i, j} \mathrel{+}= \sum \mathbb{I}(S_t = i \land S_{t+1} = j)$$

Penggunaan memori: **$< 10\text{ MB}$**!


In [ ]:
model_markov = MarkovChainBaseline()

print("Melatih Model Markov Chain secara streaming pada data Training (2016–2022)...")
for i, batch_df in enumerate(iter_panel_batches(split='train'), 1):
    s_t = batch_df['state'].values
    s_t1 = batch_df[TARGET_COL].values
    model_markov.partial_fit(s_t, s_t1)
    print(f"  Batch {i}: Akumulasi transisi selesai.")

print("\nMatriks Transisi Markov Terlatih P(S_{t+1} | S_t):")
display(model_markov.get_transition_matrix().round(5))

# Simpan artefak Markov
markov_file = MODELS_DIR / 'markov_model.joblib'
joblib.dump(model_markov, markov_file)
print(f"Model Markov disimpan ke {markov_file}")


---
## 3. Model 2A — Streaming Logistic Regression (Incremental SGD)

Model regresi logistik linier yang memasukkan seluruh kovariat lingkungan ($X_t$):
- Suhu, kelembapan relatif, curah hujan harian, akumulasi hujan 3 & 7 hari, CDD, tutupan lahan, dan lag spasial tetangga.

### Efisiensi Memori Ekstrem:
Model dilatih menggunakan algoritma **Stochastic Gradient Descent online (`SGDClassifier` dengan `partial_fit`)**:
1. **Pass 1**: Menghitung rata-rata dan varians fitur secara bertahap via `StandardScaler.partial_fit`.
2. **Pass 2**: Memperbarui bobot regresi logistik secara online batch demi batch dengan penalti regularisasi dan bobot kelas terkalibrasi.
3. Memori RAM yang digunakan selalu stabil di **$< 200\text{ MB}$**!


In [ ]:
# Inisialisasi dan pelatihan Streaming Logistic Regression
model_lr = StreamingLogisticRegression(penalty='elasticnet', alpha=1e-4)

# Menjalankan pelatihan streaming melintasi seluruh batch training
model_lr.train_streaming(feature_cols=FEATURE_COLS, target_col=TARGET_COL)

# Simpan model
model_lr.save(MODELS_DIR / 'logistic_regression.joblib')


---
## 4. Model 2B — Gradient Boosted Decision Trees (LightGBM Memory-Efficient)

LightGBM adalah algoritma *state-of-the-art* untuk pemodelan data tabular non-linier.
Namun, karena dataset training memiliki 12,3 juta baris, memasukkan seluruhnya sekaligus dapat membebani RAM laptop.

### 💡 Strategi Subsampling Terstratifikasi Cerdas:
- Kebakaran adalah kejadian langka (kelas $S=1$ hanya ~0,7%).
- Kita mempertahankan **100% seluruh kejadian positif ($S=1$)** dari seluruh tahun pelatihan.
- Kita mengambil sampel terstratifikasi acak **10% dari kelas negatif ($S=0$)**.
- Bobot sampel (*instance weights*) disesuaikan secara matematis ($w_{neg} = 10.0, w_{pos} = 1.0$) sehingga kalibrasi probabilitas dan prior empiris tetap identik.
- Ukuran data training menyusut dari 12,3 juta menjadi **~1,2 juta baris**, sehingga LightGBM dapat dilatih dengan sangat cepat (< 1 menit) dan aman di laptop (RAM $< 1\text{ GB}$), dengan fitur *Early Stopping* berbasis validation set (2023–2024).


In [ ]:
# Melatih LightGBM dengan subsampling negatif terstratifikasi
model_lgb = train_lightgbm_memory_efficient(
    feature_cols=FEATURE_COLS,
    target_col=TARGET_COL,
    neg_subsample_ratio=0.10,
    random_state=42
)


---
## 5. Analisis Tingkat Kepentingan Fitur (*Feature Importance*)
Variabel lingkungan dan histori manakah yang paling menentukan apakah suatu grid akan mengalami kebakaran esok hari?


In [ ]:
# Plot Feature Importance
feat_imp = pd.DataFrame({
    'Fitur': FEATURE_COLS,
    'Importance': model_lgb.feature_importances_
}).sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(feat_imp['Fitur'], feat_imp['Importance'], color='#0284C7', edgecolor='black', alpha=0.85)
ax.set_title('Tingkat Kepentingan Fitur (Feature Importance - Gain) LightGBM', fontweight='bold')
ax.set_xlabel('Total Gain')
plt.tight_layout()
plt.show()


---
## 6. Rangkuman Pelatihan Model

Seluruh 4 model pada Blueprint telah berhasil dibangun dan disimpan ke `outputs/models/`:
1. `PersistenceBaseline` (Model 0 - Baseline naif)
2. `MarkovChainBaseline` (Model 1 - Rantai Markov $P(S_{t+1}|S_t)$)
3. `StreamingLogisticRegression` (Model 2A - Regresi Logistik Linier)
4. `LightGBMClassifier` (Model 2B - Gradient Boosted Trees Non-Linier)

👉 **Langkah Selanjutnya**: Buka dan jalankan **`04_decision_layer_and_evaluation.ipynb`** untuk mengeksekusi Lapisan Keputusan (*Decision Layer*), menguji performa pada Test Set (2025–2026), menghitung Recall@K dan Precision@K, serta menjawab Research Questions (RQ1, RQ2, RQ3).
